In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# --------------------------------------------------
# STEP 1: DOCUMENT-TERM MATRIX CONSTRUCTION
# --------------------------------------------------

documents = [
    "machine learning uses data for prediction",
    "machine learning algorithms process data",
    "data science uses machine learning",
    "teachers explain machine learning concepts",
    "instructors teach students about computers",
    "students learn computer science and data"
]


# Function to create Document-Term Matrix
def makeDocumentTermMatrix(data):

    # Create vocabulary containing unique words
    words = []

    for doc in data:
        for word in doc.lower().split():
            if word not in words:
                words.append(word)

    # Create matrix of zeros
    # Rows = Words, Columns = Documents
    matrix = np.zeros((len(words), len(data)))

    # Fill word counts
    for i in range(len(words)):
        for j in range(len(data)):

            word_count = data[j].lower().split().count(words[i])

            matrix[i][j] = word_count

    return words, matrix


words, X = makeDocumentTermMatrix(documents)

print("Vocabulary:")
print(words)

print("\nDocument-Term Matrix:")
print(pd.DataFrame(X, index=words))


# --------------------------------------------------
# STEP 1.2: LOG-ENTROPY NORMALIZATION
# --------------------------------------------------

def logEntropyNormalization(X):

    # Number of documents
    n = X.shape[1]

    # Local weight calculation
    # log(1 + frequency)
    local = np.log10(1 + X)

    # Calculate total frequency of each word
    row_sum = np.sum(X, axis=1)

    # Calculate probability of each word in each document
    probability = np.zeros(X.shape)

    for i in range(X.shape[0]):

        if row_sum[i] != 0:
            probability[i] = X[i] / row_sum[i]

    # Calculate entropy of each word
    entropy = np.zeros(X.shape[0])

    for i in range(X.shape[0]):

        for j in range(n):

            if probability[i][j] > 0:

                entropy[i] += probability[i][j] * np.log10(
                    probability[i][j]
                )

    # Global weight
    global_weight = 1 + entropy / np.log10(n)

    # Final normalized matrix
    normalized = local * global_weight[:, np.newaxis]

    return normalized


A = logEntropyNormalization(X)

print("\nNormalized Matrix:")
print(pd.DataFrame(A, index=words))


# --------------------------------------------------
# STEP 2: DIMENSIONALITY REDUCTION USING SVD
# --------------------------------------------------

# Perform Singular Value Decomposition
U, S, VT = np.linalg.svd(A, full_matrices=False)

# Select exactly 2 latent concepts
k = 2

U_k = U[:, :k]
S_k = S[:k]
VT_k = VT[:k, :]

# Convert singular values into diagonal matrix
Sigma_k = np.diag(S_k)

print("\nU Matrix:")
print(U_k)

print("\nSigma Matrix:")
print(Sigma_k)

print("\nVT Matrix:")
print(VT_k)


# --------------------------------------------------
# STEP 2.2: PROJECT WORDS INTO LATENT SPACE
# --------------------------------------------------

# Project words into 2-dimensional concept space
projected_words = A @ VT_k.T

print("\nProjected Word Coordinates:")

projected_df = pd.DataFrame(
    projected_words,
    index=words,
    columns=["Concept 1", "Concept 2"]
)

print(projected_df)


# --------------------------------------------------
# STEP 3: FIND CLOSEST WORDS USING COSINE SIMILARITY
# --------------------------------------------------

def find_closest_words(target_word):

    # Check whether word exists
    if target_word not in words:
        print("Word not found in vocabulary")
        return

    # Get index of target word
    index = words.index(target_word)

    # Get target word's vector
    target_vector = projected_words[index]

    similarities = []

    # Compare target word with every other word
    for i in range(len(words)):

        if i == index:
            continue

        other_vector = projected_words[i]

        # Calculate dot product
        dot_product = np.dot(target_vector, other_vector)

        # Calculate vector magnitudes
        magnitude1 = np.sqrt(np.sum(target_vector ** 2))
        magnitude2 = np.sqrt(np.sum(other_vector ** 2))

        # Calculate cosine similarity
        if magnitude1 != 0 and magnitude2 != 0:

            similarity = dot_product / (magnitude1 * magnitude2)

        else:
            similarity = 0

        similarities.append((words[i], similarity))

    # Sort in descending order of similarity
    similarities.sort(key=lambda x: x[1], reverse=True)

    print("\nClosest words to:", target_word)

    for word, score in similarities[:3]:

        print(word, ":", round(score, 4))


# --------------------------------------------------
# STEP 3.2: TEST THE MODEL
# --------------------------------------------------

find_closest_words("machine")

find_closest_words("data")


# --------------------------------------------------
# VISUALIZATION OF LATENT CONCEPT SPACE
# --------------------------------------------------

plt.figure(figsize=(10, 7))

for i, word in enumerate(words):

    x = projected_words[i][0]
    y = projected_words[i][1]

    plt.scatter(x, y)
    plt.text(x + 0.01, y + 0.01, word)

plt.xlabel("Latent Concept 1")
plt.ylabel("Latent Concept 2")
plt.title("Latent Semantic Analysis - Word Mapping")

plt.grid()
plt.show()
